In [0]:
raw_df = spark.table("career_flow_engine.bronze.careerflow_raw")
display(raw_df)

In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, lit

# Flatten the raw data
flatten_df = raw_df.select(
    'company_name',
    'country',
    'employment_type',
    'industries',
    'job_function',
    'job_id',
    'job_title',
    'num_applicants',
    col('scraped_at').cast('timestamp').alias('scraped_at'),
    'scraper_version',
    'seniority_level',
    'time_posted',
    explode('skills').alias('skills')
)

In [0]:
# Check for duplicate rows in flatten_df based on all columns
duplicate_df = flatten_df.groupBy(
    'company_name',
    'country',
    'employment_type',
    'industries',
    'job_function',
    'job_id',
    'job_title',
    'num_applicants',
    'scraped_at',
    'scraper_version',
    'seniority_level',
    'time_posted',
    'skills'
).count().filter(col("count") > 1)

display(duplicate_df)

In [0]:
from pyspark.sql.functions import udf, current_timestamp,regexp_extract, col
from pyspark.sql.types import TimestampType
import re
from datetime import timedelta, datetime

def parse_relative_time(time_str):
    now = datetime.now()
    if not time_str or not isinstance(time_str, str):
        return None
    time_str = time_str.lower()
    match = re.match(r"(\d+)\s+(day|hour|week|month|year)s?\s+ago", time_str)
    if match:
        value, unit = int(match.group(1)), match.group(2)
        if unit == "day":
            return now - timedelta(days=value)
        elif unit == "hour":
            return now - timedelta(hours=value)
        elif unit == "week":
            return now - timedelta(weeks=value)
        elif unit == "month":
            # Approximate a month as 30 days
            return now - timedelta(days=30 * value)
        elif unit == "year":
            # Approximate a year as 365 days
            return now - timedelta(days=365 * value)
    return None

parse_relative_time_udf = udf(parse_relative_time, TimestampType())

transformed_df = flatten_df.withColumn(
    "time_posted_ts", parse_relative_time_udf(col("time_posted")))\
    .withColumn("effective_start", col("scraped_at")) \
    .withColumn("effective_end", lit(None).cast("timestamp")) \
    .withColumn("is_current", col("scraped_at").isNotNull())\
    .withColumn("num_applicants_int",regexp_extract(col("num_applicants"), r"(\d+)", 1).cast("int"))
display(transformed_df)

In [0]:
transformed_df.write.format("delta").mode("append").saveAsTable("career_flow_engine.silver.careerflow_jobs_cleansed")

In [0]:
%sql
select * from career_flow_engine.silver.careerflow_jobs_cleansed

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

deduped_df = (
    transformed_df
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("job_id", "skills").orderBy(F.desc("scraped_at"))
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

deduped_df.createOrReplaceTempView("cdc_updates")
display(deduped_df)

In [0]:
# Define target table name
target_table = "career_flow_engine.silver.careerflow_jobs_cleansed"

# Merge for SCD Type 2 incremental load
deduped_df.createOrReplaceTempView("cdc_updates")

merge_sql = f"""
MERGE INTO {target_table} AS target
USING cdc_updates AS source
ON target.job_id = source.job_id AND target.skills = source.skills
WHEN MATCHED AND target.is_current = true AND (
    target.company_name <> source.company_name OR
    target.country <> source.country OR
    target.employment_type <> source.employment_type OR
    target.industries <> source.industries OR
    target.job_function <> source.job_function OR
    target.job_title <> source.job_title OR
    target.num_applicants <> source.num_applicants OR
    target.scraper_version <> source.scraper_version OR
    target.seniority_level <> source.seniority_level OR
    target.time_posted <> source.time_posted
) THEN
  UPDATE SET
    target.effective_end = source.scraped_at,
    target.is_current = false
WHEN NOT MATCHED THEN
  INSERT (
    company_name,
    country,
    employment_type,
    industries,
    job_function,
    job_id,
    job_title,
    num_applicants,
    scraped_at,
    scraper_version,
    seniority_level,
    time_posted,
    skills,
    effective_start,
    effective_end,
    is_current
  )
  VALUES (
    source.company_name,
    source.country,
    source.employment_type,
    source.industries,
    source.job_function,
    source.job_id,
    source.job_title,
    source.num_applicants,
    source.scraped_at,
    source.scraper_version,
    source.seniority_level,
    source.time_posted,
    source.skills,
    source.effective_start,
    source.effective_end,
    source.is_current
  )
"""

spark.sql(merge_sql)

In [0]:
deduped_df.write.format("delta").mode("overwrite").saveAsTable("career_flow_engine.silver.careerflow_jobs_cleansed")

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr

# Define retention period
retention_days = 60

# Filter out records older than retention period
delete_sql = f"""
DELETE FROM {target_table}
WHERE effective_start < DATEADD(day, -{retention_days}, CURRENT_DATE())
"""

spark.sql(delete_sql)


# Use VACUUM to remove old files from the Delta table
# Note: VACUUM only removes files no longer referenced by the Delta table, not rows based on a date filter.
# You can specify a retention period in hours (default is 168 hours = 7 days).

# Set retention period in hours (e.g., 60 days * 24 hours)
# retention_hours = 60 * 24

# spark.sql(f"VACUUM {target_table} RETAIN {retention_hours} HOURS")

In [0]:
%sql
describe history career_flow_engine.silver.careerflow_jobs_cleansed